# Otimização de hiperparâmetros com optuna

In [ ]:
# Supondo que X (features selecionadas) e y (alvo: Resistencia_MPa) já estejam definidos
# Usaremos o conjunto de treino (`X_train`, `y_train`) para realizar a validação cruzada interna
# Copia para evitar modificações acidentais
X_cv = X_train.copy()
y_cv = y_train.copy()

def objective(trial):
    # 1. Definindo o Espaço de Busca (Search Space)
    params = {
        "booster": "dart",
        "sample_type": "uniform",
        "normalize_type": "tree",
        "eval_metric": "rmse",
        "random_state": 42,
        "n_jobs": -1, # Usa todos os núcleos do processador

        # Parâmetros de Dropout (DART)
        "rate_drop": trial.suggest_float("rate_drop", 0.05, 0.20),
        "n_estimators": trial.suggest_int("n_estimators", 300, 800, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),

        # Estrutura da Árvore e Regularização
        "max_depth": trial.suggest_int("max_depth", 2, 4), # Mantemos raso (2 a 4)
        "min_child_weight": trial.suggest_int("min_child_weight", 8, 20),
        "subsample": trial.suggest_float("subsample", 0.5, 0.8),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "reg_lambda": trial.suggest_float("reg_lambda", 10.0, 25.0)
    }

    # 2. Configurando a Validação Cruzada Robusta
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []

    for train_idx, val_idx in kf.split(X_cv):
        # O .iloc garante que funciona com Pandas DataFrames
        X_tr, X_val = X_cv.iloc[train_idx], X_cv.iloc[val_idx]
        y_tr, y_val = y_cv.iloc[train_idx], y_cv.iloc[val_idx]

        model = XGBRegressor(**params)

        # Treino SEM early stopping (obrigatório para não bugar o DART)
        model.fit(X_tr, y_tr, verbose=False)

        # Predição e cálculo do erro (RMSE)
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmse_scores.append(rmse)

    # 3. O Optuna tentará MINIMIZAR o RMSE médio dos 5 Folds
    return np.mean(rmse_scores)

# ==========================================
# EXECUTANDO O ESTUDO
# ==========================================

# Cria o estudo. A direção é "minimize" porque queremos o menor RMSE possível.
study = optuna.create_study(direction="minimize", study_name="Tuning_DART_Concreto")

# Como o DART demora mais, 30 a 50 trials já trazem resultados fantásticos 
# com a busca bayesiana. (Pode levar alguns minutos)
print("Iniciando a otimização com Optuna...")
study.optimize(objective, n_trials=30)

print("\n" + "="*50)
print("🏆 OTIMIZAÇÃO CONCLUÍDA")
print("="*50)
print(f"Melhor RMSE alcançado (Cross-Validation): {study.best_value:.4f}")
print("Melhores Hiperparâmetros encontrados:")
for key, value in study.best_params.items():
    print(f"  '{key}': {value},")